In [1]:
import os
import sys
import time
import numpy as np
import xarray as xr
from glob import glob

In [2]:
import pandas as pd
from tqdm import tqdm

In [3]:
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
from scipy.stats import norm

In [5]:
from scipy.interpolate import interp1d

### Get post-processed fcsts

In [6]:
EPS = 1e-6
N_QUANTILES = 50
MIN_SAMPLES = 10

In [7]:
years_train = np.arange(1959, 1999)
years_verif = np.arange(1999, 2021)

In [8]:
ds_input = xr.open_dataset('/glade/derecho/scratch/ksha/EPRI_AnEn/baseline_CESM_member_20260409.nc')
ds_target = xr.open_dataset('/glade/derecho/scratch/ksha/EPRI_AnEn/input_AnEn_ERA5_20260221.nc')
 
list_keys_CESM = list(ds_input.keys())
list_keys_ERA5 = []
rename_keys = {}
for varname in list_keys_CESM:
    list_keys_ERA5.append('ERA5' + varname[4:])
    rename_keys[varname] = 'ERA5' + varname[4:]
 
ds_input = ds_input.rename(rename_keys)
ds_target = ds_target[list_keys_ERA5]
 
var_precip = [
    'ERA5_precip_annual_max_30d',
    'ERA5_precip_max_daily',
    'ERA5_precip_mean',
]
var_others = sorted(set(list_keys_ERA5) - set(var_precip))
 
# Split train / verif
ds_input_train  = ds_input.sel(gen_date=years_train)
ds_target_train = ds_target.sel(gen_date=years_train)
ds_input_verif  = ds_input.sel(gen_date=years_verif)
ds_target_verif = ds_target.sel(gen_date=years_verif)
 
n_site = ds_input_train.sizes['site']
n_lead = ds_input_train.sizes['lead_time']
n_train = ds_input_train.sizes['gen_date']
n_member = ds_input_train.sizes['member']
n_verif = ds_input_verif.sizes['gen_date']
 
print(f"Sites: {n_site}, Lead years: {n_lead}, Train: {n_train}, Verif: {n_verif}")
print(f"Members: {n_member}")
print(f"Non-precip variables: {var_others}")
print(f"Precip variables: {var_precip}\n")

Sites: 5, Lead years: 10, Train: 40, Verif: 22
Members: 20
Non-precip variables: ['ERA5_t2m_annual_max_30d', 'ERA5_t2m_max_daily', 'ERA5_t2m_max_hour', 'ERA5_t2m_mean', 'ERA5_t2m_min_daily', 'ERA5_t2m_min_hour']
Precip variables: ['ERA5_precip_annual_max_30d', 'ERA5_precip_max_daily', 'ERA5_precip_mean']



In [9]:
q_levels = np.linspace(0.5 / N_QUANTILES, 1.0 - 0.5 / N_QUANTILES, N_QUANTILES)

## BMA core functions

In [10]:
def fit_bma_equal(ens, obs):
    """
    Fit BMA under the equal member assumption.
 
    Predictive: p(y) = (1/K) Σ N(y; a + b·fk, σ²)
 
    All members share (a, b). Weights fixed at 1/K.
    Only 3 parameters: a, b, σ².
 
    Parameters
    ----------
    ens : (n_time, n_member)
    obs : (n_time,)
 
    Returns
    -------
    a, b   : shared bias correction
    sigma2 : shared variance
    """
    valid = ~np.isnan(obs)
    if valid.sum() < MIN_SAMPLES:
        return np.nan, np.nan, np.nan
 
    ob = obs[valid]
    en = ens[valid]
    N, K = en.shape
 
    # Pool all (time, member) pairs for shared OLS
    f_flat = en.ravel()
    o_flat = np.repeat(ob, K)
 
    valid_flat = ~np.isnan(f_flat)
    if valid_flat.sum() < MIN_SAMPLES:
        return np.nan, np.nan, np.nan
 
    f = f_flat[valid_flat]
    o = o_flat[valid_flat]
 
    f_bar = f.mean()
    o_bar = o.mean()
    cov = ((f - f_bar) * (o - o_bar)).mean()
    var = ((f - f_bar) ** 2).mean()
 
    b = cov / max(var, EPS)
    a = o_bar - b * f_bar
 
    # Shared variance: mean squared residual across all (time, member)
    ens_corr = a + b * en
    resid_sq = (ob[:, None] - ens_corr) ** 2
    sigma2 = np.nanmean(resid_sq)
    sigma2 = max(sigma2, EPS)
 
    return a, b, sigma2
 
 
def apply_bma_equal(ens, a, b):
    """Apply shared bias correction: a + b * fk."""
    return a + b * ens
 
 
def bma_equal_mean(ens_corrected):
    """BMA predictive mean = mean of corrected members."""
    return np.nanmean(ens_corrected, axis=-1)
 
 
def crps_bma_equal(ens_corrected, sigma2, obs):
    """
    Closed-form CRPS for equal-weight Gaussian mixture.
 
    p(y) = (1/K) Σ N(y; mk, σ²)
 
    CRPS = (1/K) Σ A(mk,σ,y) - (1/2K²) ΣΣ E|Xk-Xj|
    """
    sigma = np.sqrt(max(sigma2, EPS))
    n_time, K = ens_corrected.shape
 
    def _A(mu, sig, y):
        z = (y - mu) / sig
        return sig * (z * (2.0 * norm.cdf(z) - 1.0) + 2.0 * norm.pdf(z) - 1.0 / np.sqrt(np.pi))
 
    # Term 1
    term1 = np.zeros(n_time)
    for k in range(K):
        term1 += _A(ens_corrected[:, k], sigma, obs)
    term1 /= K
 
    # Term 2
    sigma_root2 = sigma * np.sqrt(2.0)
    term2 = np.zeros(n_time)
    for k in range(K):
        for j in range(K):
            diff = ens_corrected[:, k] - ens_corrected[:, j]
            z_kj = diff / sigma_root2
            E_abs = sigma_root2 * (
                2.0 * norm.pdf(z_kj) + z_kj * (2.0 * norm.cdf(z_kj) - 1.0)
            )
            term2 += E_abs
    term2 /= (2.0 * K * K)
 
    return term1 - term2
 
 
def crps_raw_ensemble(ens, obs):
    """Fair empirical CRPS from raw ensemble members."""
    ens_sorted = np.sort(ens, axis=1)
    n_time, n_mem = ens_sorted.shape
    term1 = np.mean(np.abs(ens_sorted - obs[:, None]), axis=1)
    weights = 2.0 * np.arange(1, n_mem + 1) - n_mem - 1.0
    term2 = np.sum(weights[None, :] * ens_sorted, axis=1) / (n_mem * (n_mem - 1))
    return term1 - term2
 
 
def compute_acc(forecast, observation, climatology):
    """Anomaly Correlation Coefficient."""
    f_anom = forecast - climatology
    o_anom = observation - climatology
    valid = ~np.isnan(f_anom) & ~np.isnan(o_anom)
    if valid.sum() < 3:
        return np.nan
    f_a, o_a = f_anom[valid], o_anom[valid]
    num = np.sum(f_a * o_a)
    den = np.sqrt(np.sum(f_a ** 2) * np.sum(o_a ** 2))
    return num / den if den > EPS else np.nan

### Quantile transform functions

In [11]:
def build_quantile_table(samples):
    """Build empirical quantile table from 1D samples."""
    valid = samples[~np.isnan(samples)]
    if len(valid) < MIN_SAMPLES:
        return np.full(N_QUANTILES, np.nan), np.nan
    q_values = np.quantile(valid, q_levels).astype(np.float32)
    wet_frac = np.float32(np.mean(valid > 0.0))
    return q_values, wet_frac
 
 
def extrapolate_upper_tail(q_vals, q_levs):
    """Linearly extrapolate to quantile level 1.0."""
    v1, v2 = q_vals[-2], q_vals[-1]
    l1, l2 = q_levs[-2], q_levs[-1]
    slope = (v2 - v1) / max(l2 - l1, EPS)
    v_max = max(v2 + slope * (1.0 - l2), v2)
    return np.append(q_vals, np.float32(v_max)), np.append(q_levs, 1.0)
 
 
def precip_to_gaussian(values, q_values, wet_frac, rng=None):
    """Map precipitation → Gaussian via NQT."""
    n = len(values)
    gaussian = np.full(n, np.nan, dtype=np.float32)
 
    if np.all(np.isnan(q_values)):
        return gaussian
    if rng is None:
        rng = np.random.default_rng()
 
    valid = ~np.isnan(values)
    vals = np.maximum(values[valid], 0.0)
 
    dry_frac = 1.0 - wet_frac
    u = np.full(len(vals), np.nan, dtype=np.float64)
 
    is_zero = vals <= 0.0
    is_wet = ~is_zero
 
    # Zeros: jitter in [EPS, dry_frac]
    if dry_frac > 0 and is_zero.any():
        u[is_zero] = rng.uniform(EPS, max(dry_frac, 2 * EPS), size=is_zero.sum())
 
    # Positives: interpolate via wet quantiles
    if is_wet.any():
        wet_q_mask = q_values > 0.0
        if wet_q_mask.sum() >= 2:
            q_wet = q_values[wet_q_mask]
            l_wet = q_levels[wet_q_mask]
            q_ext, l_ext = extrapolate_upper_tail(q_wet, l_wet)
 
            q_unique, idx_unique = np.unique(q_ext, return_index=True)
            ql_unique = l_ext[idx_unique]
 
            if len(q_unique) >= 2:
                interp_func = interp1d(
                    q_unique, ql_unique,
                    kind='linear', bounds_error=False,
                    fill_value=(ql_unique[0], ql_unique[-1]),
                )
                u_wet_raw = interp_func(vals[is_wet])
                l_min, l_max = ql_unique[0], ql_unique[-1]
                u[is_wet] = dry_frac + (1.0 - dry_frac) * (
                    (u_wet_raw - l_min) / max(l_max - l_min, EPS)
                )
            else:
                u[is_wet] = dry_frac + 0.5 * (1.0 - dry_frac)
        else:
            u[is_wet] = dry_frac + 0.5 * (1.0 - dry_frac)
 
    u = np.clip(u, 1e-6, 1.0 - 1e-6)
    gaussian[valid] = norm.ppf(u).astype(np.float32)
    return gaussian
 
 
def gaussian_to_precip(gaussian_values, q_values, wet_frac):
    """Inverse NQT: Gaussian → precipitation (mm/day)."""
    shape = gaussian_values.shape
    g_flat = gaussian_values.ravel()
    precip = np.full_like(g_flat, np.nan, dtype=np.float32)
 
    if np.all(np.isnan(q_values)):
        return precip.reshape(shape)
 
    valid = ~np.isnan(g_flat)
    if not valid.any():
        return precip.reshape(shape)
 
    u = norm.cdf(g_flat[valid])
    dry_frac = 1.0 - wet_frac
    result = np.zeros_like(u, dtype=np.float32)
 
    wet_mask = u > dry_frac
    if wet_mask.any():
        wet_q_mask = q_values > 0.0
        if wet_q_mask.sum() >= 2:
            q_wet = q_values[wet_q_mask]
            l_wet = q_levels[wet_q_mask]
            q_ext, l_ext = extrapolate_upper_tail(q_wet, l_wet)
 
            q_unique, idx_unique = np.unique(q_ext, return_index=True)
            ql_unique = l_ext[idx_unique]
 
            if len(q_unique) >= 2:
                l_min, l_max = ql_unique[0], ql_unique[-1]
                u_rescaled = l_min + (l_max - l_min) * (
                    (u[wet_mask] - dry_frac) / max(1.0 - dry_frac, EPS)
                )
                inv_func = interp1d(
                    ql_unique, q_unique,
                    kind='linear', bounds_error=False,
                    fill_value=(q_unique[0], q_unique[-1]),
                )
                result[wet_mask] = np.maximum(inv_func(u_rescaled), 0.0)
            else:
                result[wet_mask] = q_values[wet_q_mask].mean()
        elif wet_q_mask.any():
            result[wet_mask] = q_values[wet_q_mask][0]
 
    precip[valid] = result.astype(np.float32)
    return precip.reshape(shape)

### BMA on 2-m Temp

In [12]:
bma_params = {}        # {var: {(site,lead): (a, b, weights, sigma2)}}

raw_train_crps = {}
bma_train_crps = {}
raw_verif_crps = {}
bma_verif_crps = {}


for var in var_others:
    print(f"\n  Fitting {var} …")

    params_var = {}
    verif_crps_raw  = np.full((n_site, n_lead, n_verif), np.nan, dtype=np.float32)
    verif_crps_bma  = np.full((n_site, n_lead, n_verif), np.nan, dtype=np.float32)

    for si in range(n_site):
        for lt in range(n_lead):

            ens = ds_input_train[var].isel(site=si, lead_time=lt).values
            obs = ds_target_train[var].isel(site=si, lead_time=lt).values

            # Fit: 3 parameters only
            a, b, sigma2 = fit_bma_equal(ens, obs)
            params_var[(si, lt)] = (a, b, sigma2)

            if np.isnan(sigma2):
                continue

            # Verify
            ens_v = ds_input_verif[var].isel(site=si, lead_time=lt).values
            obs_v = ds_target_verif[var].isel(site=si, lead_time=lt).values

            ens_corr_v = apply_bma_equal(ens_v, a, b)
            verif_crps_bma[si, lt] = crps_bma_equal(ens_corr_v, sigma2, obs_v)
            verif_crps_raw[si, lt] = crps_raw_ensemble(ens_v, obs_v)

    bma_params[var] = params_var
    raw_verif_crps[var] = verif_crps_raw
    bma_verif_crps[var] = verif_crps_bma

    r = np.nanmean(verif_crps_raw)
    e = np.nanmean(verif_crps_bma)
    imp = 100 * (r - e) / r if r > 0 else 0
    print(f"    Verif CRPS — raw: {r:.4f}, BMA: {e:.4f}, gain: {imp:+.1f}%")


  Fitting ERA5_t2m_annual_max_30d …
    Verif CRPS — raw: 2.7679, BMA: 0.2979, gain: +89.2%

  Fitting ERA5_t2m_max_daily …
    Verif CRPS — raw: 2.7905, BMA: 0.1627, gain: +94.2%

  Fitting ERA5_t2m_max_hour …
    Verif CRPS — raw: 6.6752, BMA: 0.2472, gain: +96.3%

  Fitting ERA5_t2m_mean …
    Verif CRPS — raw: 1.1827, BMA: 0.1538, gain: +87.0%

  Fitting ERA5_t2m_min_daily …
    Verif CRPS — raw: 1.3480, BMA: 0.1948, gain: +85.6%

  Fitting ERA5_t2m_min_hour …
    Verif CRPS — raw: 4.1329, BMA: 0.0744, gain: +98.2%


###  NQT + Gaussian BMA for precip variables

In [13]:
precip_qtables = {}
rng = np.random.default_rng(42)
 
for var in var_precip:
    print(f"\n  Processing {var} …")
 
    params_var = {}
    qtables_var = {}
    t_crps_raw = np.full((n_site, n_lead, n_train), np.nan, dtype=np.float32)
    t_crps_bma = np.full((n_site, n_lead, n_train), np.nan, dtype=np.float32)
    v_crps_raw = np.full((n_site, n_lead, n_verif), np.nan, dtype=np.float32)
    v_crps_bma = np.full((n_site, n_lead, n_verif), np.nan, dtype=np.float32)
 
    for si in range(n_site):
        for lt in range(n_lead):
            ens = np.maximum(
                ds_input_train[var].isel(site=si, lead_time=lt).values, 0.0)
            obs = np.maximum(
                ds_target_train[var].isel(site=si, lead_time=lt).values, 0.0)
 
            # Quantile tables
            qt_ens, wf_ens = build_quantile_table(ens.ravel())
            qt_obs, wf_obs = build_quantile_table(obs)
            qtables_var[(si, lt)] = (qt_ens, wf_ens, qt_obs, wf_obs)
 
            if np.isnan(wf_ens) or np.isnan(wf_obs):
                continue
 
            # Transform to Gaussian
            ens_gauss = np.full_like(ens, np.nan, dtype=np.float64)
            for mi in range(n_member):
                ens_gauss[:, mi] = precip_to_gaussian(ens[:, mi], qt_ens, wf_ens, rng=rng)
            obs_gauss = precip_to_gaussian(obs, qt_obs, wf_obs, rng=rng)
 
            # Fit BMA in Gaussian space
            a, b, sigma2 = fit_bma_equal(ens_gauss, obs_gauss)
            params_var[(si, lt)] = (a, b, sigma2)
 
            if np.isnan(sigma2):
                continue
 
            # Train CRPS
            ens_corr = apply_bma_equal(ens_gauss, a, b)
            t_crps_bma[si, lt] = crps_bma_equal(ens_corr, sigma2, obs_gauss)
            t_crps_raw[si, lt] = crps_raw_ensemble(ens_gauss, obs_gauss)
 
            # Verification
            ens_v = np.maximum(
                ds_input_verif[var].isel(site=si, lead_time=lt).values, 0.0)
            obs_v = np.maximum(
                ds_target_verif[var].isel(site=si, lead_time=lt).values, 0.0)
 
            ens_gauss_v = np.full_like(ens_v, np.nan, dtype=np.float64)
            for mi in range(n_member):
                ens_gauss_v[:, mi] = precip_to_gaussian(
                    ens_v[:, mi], qt_ens, wf_ens, rng=rng)
            obs_gauss_v = precip_to_gaussian(obs_v, qt_obs, wf_obs, rng=rng)
 
            ens_corr_v = apply_bma_equal(ens_gauss_v, a, b)
            v_crps_bma[si, lt] = crps_bma_equal(ens_corr_v, sigma2, obs_gauss_v)
            v_crps_raw[si, lt] = crps_raw_ensemble(ens_gauss_v, obs_gauss_v)
 
    bma_params[var] = params_var
    precip_qtables[var] = qtables_var
    raw_train_crps[var] = t_crps_raw
    bma_train_crps[var] = t_crps_bma
    raw_verif_crps[var] = v_crps_raw
    bma_verif_crps[var] = v_crps_bma
 
    r = np.nanmean(v_crps_raw)
    e = np.nanmean(v_crps_bma)
    imp = 100 * (r - e) / r if r > 0 else 0
    print(f"    Verif CRPS (Gaussian space) — raw: {r:.4f}, BMA: {e:.4f}, gain: {imp:+.1f}%")


  Processing ERA5_precip_annual_max_30d …
    Verif CRPS (Gaussian space) — raw: 0.8393, BMA: 0.0921, gain: +89.0%

  Processing ERA5_precip_max_daily …
    Verif CRPS (Gaussian space) — raw: 0.7113, BMA: -0.0287, gain: +104.0%

  Processing ERA5_precip_mean …
    Verif CRPS (Gaussian space) — raw: 0.8037, BMA: 0.0588, gain: +92.7%


### Ensemble dressing

In [14]:
N_ens = 50
rng_dress = np.random.default_rng(123)
ds_results = {}
 
# Non-precip
for var in var_others:
    print(f"\n  Dressing {var} …")
    dressed = np.full((n_site, n_lead, n_verif, N_ens), np.nan, dtype=np.float32)
 
    for si in range(n_site):
        for lt in range(n_lead):
            a, b, sigma2 = bma_params[var][(si, lt)]
            if np.isnan(sigma2):
                continue
 
            sigma = np.sqrt(sigma2)
            ens_v = ds_input_verif[var].isel(site=si, lead_time=lt).values
            ens_corr_v = apply_bma_equal(ens_v, a, b)   # (n_verif, K)
 
            for t in range(n_verif):
                components = rng_dress.integers(0, n_member, size=N_ens)
                means = ens_corr_v[t, components]
                dressed[si, lt, t, :] = rng_dress.normal(loc=means, scale=sigma)
 
    ds_results[var] = dressed
 
# Precip
for var in var_precip:
    print(f"\n  Dressing {var} (with inverse NQT) …")
    dressed = np.full((n_site, n_lead, n_verif, N_ens), np.nan, dtype=np.float32)
 
    for si in range(n_site):
        for lt in range(n_lead):
            a, b, sigma2 = bma_params[var][(si, lt)]
            if np.isnan(sigma2):
                continue
 
            qt_ens, wf_ens, qt_obs, wf_obs = precip_qtables[var][(si, lt)]
            if np.isnan(wf_obs):
                continue
 
            sigma = np.sqrt(sigma2)
            ens_v = np.maximum(
                ds_input_verif[var].isel(site=si, lead_time=lt).values, 0.0)
 
            # Transform and bias-correct
            ens_gauss_v = np.full_like(ens_v, np.nan, dtype=np.float64)
            for mi in range(n_member):
                ens_gauss_v[:, mi] = precip_to_gaussian(
                    ens_v[:, mi], qt_ens, wf_ens, rng=rng_dress)
            ens_corr_v = apply_bma_equal(ens_gauss_v, a, b)
 
            for t in range(n_verif):
                components = rng_dress.integers(0, n_member, size=N_ens)
                means = ens_corr_v[t, components]
                z = rng_dress.normal(loc=means, scale=sigma)
                dressed[si, lt, t, :] = gaussian_to_precip(
                    z.astype(np.float32), qt_obs, wf_obs)
 
    ds_results[var] = dressed


  Dressing ERA5_t2m_annual_max_30d …

  Dressing ERA5_t2m_max_daily …

  Dressing ERA5_t2m_max_hour …

  Dressing ERA5_t2m_mean …

  Dressing ERA5_t2m_min_daily …

  Dressing ERA5_t2m_min_hour …

  Dressing ERA5_precip_annual_max_30d (with inverse NQT) …

  Dressing ERA5_precip_max_daily (with inverse NQT) …

  Dressing ERA5_precip_mean (with inverse NQT) …


### Save

In [18]:
# print("\n" + "=" * 60)
# print("CRPS Summary (verification period)")
# print("=" * 60)
# print(f"{'Variable':<35} {'Raw CRPS':>10} {'BMA CRPS':>10} {'Gain':>8}")
# print("-" * 65)
# for var in all_vars:
#     r = np.nanmean(raw_verif_crps[var])
#     e = np.nanmean(bma_verif_crps[var])
#     imp = 100 * (r - e) / r if r > 0 else 0
#     tag = "(Gauss)" if var in var_precip else ""
#     print(f"  {var} {tag:<8} {r:>10.4f} {e:>10.4f} {imp:>+7.1f}%")
 
# print(f"\n{'Variable':<35} {'Raw ACC':>10} {'BMA ACC':>10} {'Diff':>8}")
# print("-" * 65)
# for var in all_vars:
#     r = np.nanmean(acc_raw[var])
#     e = np.nanmean(acc_bma[var])
#     print(f"  {var:<33} {r:>10.4f} {e:>10.4f} {e - r:>+8.4f}")
 
# # Per lead time
# print(f"\nPer-lead CRPS (averaged over sites):")
# print(f"{'Lead':<6}", end="")
# for var in all_vars:
#     short = var.replace('ERA5_', '')[:10]
#     print(f" {short:>12}", end="")
# print()
# print("-" * (6 + 13 * len(all_vars)))
# for lt in range(n_lead):
#     print(f"  {lt:<4}", end="")
#     for var in all_vars:
#         r = np.nanmean(raw_verif_crps[var][:, lt])
#         e = np.nanmean(bma_verif_crps[var][:, lt])
#         imp = 100 * (r - e) / r if r > 0 else 0
#         print(f"  {imp:>+8.1f}%  ", end="")
#     print()
 
# # Per site
# print(f"\nPer-site CRPS (averaged over leads):")
# print(f"{'Site':<6}", end="")
# for var in all_vars:
#     short = var.replace('ERA5_', '')[:10]
#     print(f" {short:>12}", end="")
# print()
# print("-" * (6 + 13 * len(all_vars)))
# for si in range(n_site):
#     print(f"  {si:<4}", end="")
#     for var in all_vars:
#         r = np.nanmean(raw_verif_crps[var][si])
#         e = np.nanmean(bma_verif_crps[var][si])
#         imp = 100 * (r - e) / r if r > 0 else 0
#         print(f"  {imp:>+8.1f}%  ", end="")
#     print()
 
# # # ── Save ──
# # coeff_data = {}
# # for var in all_vars:
# #     for si in range(n_site):
# #         for lt in range(n_lead):
# #             pass  # params stored in bma_params dict
# #     a_arr = np.full((n_site, n_lead), np.nan, dtype=np.float32)
# #     b_arr = np.full((n_site, n_lead), np.nan, dtype=np.float32)
# #     s_arr = np.full((n_site, n_lead), np.nan, dtype=np.float32)
# #     for si in range(n_site):
# #         for lt in range(n_lead):
# #             a, b, s2 = bma_params[var][(si, lt)]
# #             a_arr[si, lt] = a
# #             b_arr[si, lt] = b
# #             s_arr[si, lt] = s2
# #     coeff_data[f'{var}_a']      = (['site', 'lead_time'], a_arr)
# #     coeff_data[f'{var}_b']      = (['site', 'lead_time'], b_arr)
# #     coeff_data[f'{var}_sigma2'] = (['site', 'lead_time'], s_arr)
 
# # ds_coeffs = xr.Dataset(coeff_data, coords={
# #     'site': np.arange(n_site), 'lead_time': np.arange(n_lead),
# # })
# # ds_coeffs.attrs['description'] = 'BMA (equal member) coefficients: shared a, b, sigma2 per (site, lead)'
 
# # crps_data = {}
# # acc_data = {}
# # for var in all_vars:
# #     crps_data[f'{var}_raw'] = (['site', 'lead_time', 'gen_date'], raw_verif_crps[var])
# #     crps_data[f'{var}_bma'] = (['site', 'lead_time', 'gen_date'], bma_verif_crps[var])
# #     acc_data[f'{var}_raw']  = (['site', 'lead_time'], acc_raw[var])
# #     acc_data[f'{var}_bma']  = (['site', 'lead_time'], acc_bma[var])
 
# # ds_crps = xr.Dataset(crps_data, coords={
# #     'site': np.arange(n_site), 'lead_time': np.arange(n_lead), 'gen_date': years_verif,
# # })
# # ds_acc = xr.Dataset(acc_data, coords={
# #     'site': np.arange(n_site), 'lead_time': np.arange(n_lead),
# # })
 
# # dressed_data = {}
# # for var in all_vars:
# #     dressed_data[var] = (['site', 'lead_time', 'gen_date', 'member'], ds_results[var])
# # ds_dressed = xr.Dataset(dressed_data, coords={
# #     'site': np.arange(n_site), 'lead_time': np.arange(n_lead),
# #     'gen_date': years_verif, 'member': np.arange(N_ens),
# # })
# # ds_dressed.attrs['description'] = (
# #     f'Equal-weight BMA dressed ensemble (N={N_ens}). '
# #     'Non-precip: Gaussian mixture dressing. Precip: NQT + mixture dressing + inverse NQT.'
# # )
 
# # save_dir = '/glade/derecho/scratch/ksha/EPRI_AnEn'
# # ds_coeffs.to_netcdf(f'{save_dir}/bma_equal_coefficients.nc')
# # ds_crps.to_netcdf(f'{save_dir}/bma_equal_crps_verification.nc')
# # ds_acc.to_netcdf(f'{save_dir}/bma_equal_acc_verification.nc')
# # ds_dressed.to_netcdf(f'{save_dir}/bma_equal_dressed_ensemble.nc')
 
# # print(f"\nAll results saved to {save_dir}/bma_equal_*.nc")
# # print("Done.")

In [19]:
all_vars = var_others + var_precip

In [20]:
for var in all_vars:
    r = np.nanmean(raw_verif_crps[var])
    e = np.nanmean(bma_verif_crps[var])
    imp = 100 * (r - e) / r if r > 0 else 0
    space = "(Gaussian)" if var in var_precip else ""
    print(f"  {var} {space:<12} {r:>10.4f} {e:>10.4f} {imp:>+7.1f}%")
 
print("\nPer-site, per-lead breakdown for first variable:")
for var0 in all_vars:
    print(f"\n  {var0}:")
    for si in range(n_site):
        for lt in range(n_lead):
            r = np.nanmean(raw_verif_crps[var0][si, lt])
            e = np.nanmean(bma_verif_crps[var0][si, lt])
            imp = 100 * (r - e) / r if r > 0 else 0
            print(f"    Site {si}, Lead {lt}: raw={r:.4f}  bma={e:.4f}  gain={imp:+.1f}%")

  ERA5_t2m_annual_max_30d                  2.7679     0.2979   +89.2%
  ERA5_t2m_max_daily                  2.7905     0.1627   +94.2%
  ERA5_t2m_max_hour                  6.6752     0.2472   +96.3%
  ERA5_t2m_mean                  1.1827     0.1538   +87.0%
  ERA5_t2m_min_daily                  1.3480     0.1948   +85.6%
  ERA5_t2m_min_hour                  4.1329     0.0744   +98.2%
  ERA5_precip_annual_max_30d (Gaussian)       0.8393     0.0921   +89.0%
  ERA5_precip_max_daily (Gaussian)       0.7113    -0.0287  +104.0%
  ERA5_precip_mean (Gaussian)       0.8037     0.0588   +92.7%

Per-site, per-lead breakdown for first variable:

  ERA5_t2m_annual_max_30d:
    Site 0, Lead 0: raw=5.0785  bma=1.5787  gain=+68.9%
    Site 0, Lead 1: raw=5.5035  bma=1.2936  gain=+76.5%
    Site 0, Lead 2: raw=5.1964  bma=1.0362  gain=+80.1%
    Site 0, Lead 3: raw=5.2113  bma=0.9947  gain=+80.9%
    Site 0, Lead 4: raw=5.4596  bma=1.0084  gain=+81.5%
    Site 0, Lead 5: raw=5.3335  bma=0.6841  gain=+

In [21]:
clim = {}
for var in all_vars:
    clim[var] = ds_target_train[var].mean(dim='gen_date').values
 
acc_raw = {var: np.full((n_site, n_lead), np.nan, dtype=np.float32) for var in all_vars}
acc_bma = {var: np.full((n_site, n_lead), np.nan, dtype=np.float32) for var in all_vars}
 
for var in all_vars:
    is_precip = var in var_precip
 
    for si in range(n_site):
        for lt in range(n_lead):
            obs_v = ds_target_verif[var].isel(site=si, lead_time=lt).values
            clim_val = clim[var][si, lt]
 
            # Raw
            ens_v = ds_input_verif[var].isel(site=si, lead_time=lt).values
            raw_mean = np.nanmean(ens_v, axis=1)
            acc_raw[var][si, lt] = compute_acc(raw_mean, obs_v, clim_val)
 
            # BMA
            a, b, sigma2 = bma_params[var][(si, lt)]
            if np.isnan(sigma2):
                continue
 
            if is_precip:
                qt_ens, wf_ens, qt_obs, wf_obs = precip_qtables[var][(si, lt)]
                if np.isnan(wf_ens):
                    continue
                ens_v_pos = np.maximum(ens_v, 0.0)
                ens_gauss_v = np.full_like(ens_v_pos, np.nan, dtype=np.float64)
                for mi in range(n_member):
                    ens_gauss_v[:, mi] = precip_to_gaussian(
                        ens_v_pos[:, mi], qt_ens, wf_ens, rng=rng)
                ens_corr_v = apply_bma_equal(ens_gauss_v, a, b)
                bma_mean_g = bma_equal_mean(ens_corr_v)
                bma_mean = gaussian_to_precip(
                    bma_mean_g.astype(np.float32), qt_obs, wf_obs)
            else:
                ens_corr_v = apply_bma_equal(ens_v, a, b)
                bma_mean = bma_equal_mean(ens_corr_v)
 
            acc_bma[var][si, lt] = compute_acc(bma_mean, obs_v, clim_val)

# ════════════════════════════════════════════════════════════════════════
# Summary tables
# ════════════════════════════════════════════════════════════════════════

# ── Overall per variable ──
print(f"\n{'Variable':<35} {'Raw ACC':>10} {'BMA ACC':>10} {'Diff':>8}")
print("-" * 65)
for var in all_vars:
    r = np.nanmean(acc_raw[var])
    e = np.nanmean(acc_bma[var])
    print(f"  {var:<33} {r:>10.4f} {e:>10.4f} {e - r:>+8.4f}")

# ── Per lead time (averaged over sites) ──
print(f"\n{'Lead':<8}", end="")
for var in all_vars:
    short = var.replace('ERA5_', '')[:12]
    print(f" {short:>14}", end="")
print()
print("-" * (8 + 15 * len(all_vars)))

for lt in range(n_lead):
    print(f"  {lt:<6}", end="")
    for var in all_vars:
        r = np.nanmean(acc_raw[var][:, lt])
        e = np.nanmean(acc_bma[var][:, lt])
        print(f"  {r:.2f}→{e:.2f}", end="")
    print()

# ── Per site (averaged over leads) ──
print(f"\n{'Site':<8}", end="")
for var in all_vars:
    short = var.replace('ERA5_', '')[:12]
    print(f" {short:>14}", end="")
print()
print("-" * (8 + 15 * len(all_vars)))

for si in range(n_site):
    print(f"  {si:<6}", end="")
    for var in all_vars:
        r = np.nanmean(acc_raw[var][si, :])
        e = np.nanmean(acc_bma[var][si, :])
        print(f"  {r:.2f}→{e:.2f}", end="")
    print()

# # ════════════════════════════════════════════════════════════════════════
# # Save ACC results
# # ════════════════════════════════════════════════════════════════════════

# acc_data = {}
# for var in all_vars:
#     acc_data[f'{var}_raw']  = (['site', 'lead_time'], acc_raw[var])
#     acc_data[f'{var}_bma'] = (['site', 'lead_time'], acc_bma[var])

# ds_acc = xr.Dataset(
#     acc_data,
#     coords={
#         'site': np.arange(n_site),
#         'lead_time': np.arange(n_lead),
#     },
# )
# ds_acc.attrs['description'] = (
#     'Anomaly Correlation Coefficient: raw ensemble mean vs BMA calibrated mean. '
#     f'Climatology from training period ({years_train[0]}–{years_train[-1]}). '
#     f'Verification period: {years_verif[0]}–{years_verif[-1]}.'
# )

# fn_acc = '/glade/derecho/scratch/ksha/EPRI_AnEn/bma_acc_verification.nc'
# ds_acc.to_netcdf(fn_acc)
# print(f"\nSaved ACC: {fn_acc}")


Variable                               Raw ACC    BMA ACC     Diff
-----------------------------------------------------------------
  ERA5_t2m_annual_max_30d               0.3233     0.5744  +0.2510
  ERA5_t2m_max_daily                    0.3017     0.5737  +0.2720
  ERA5_t2m_max_hour                     0.6310     0.4708  -0.1602
  ERA5_t2m_mean                         0.3156     0.7856  +0.4701
  ERA5_t2m_min_daily                    0.6090     0.4118  -0.1973
  ERA5_t2m_min_hour                    -0.1738     0.2348  +0.4086
  ERA5_precip_annual_max_30d            0.0521    -0.1292  -0.1813
  ERA5_precip_max_daily                -0.0431    -0.1324  -0.0893
  ERA5_precip_mean                     -0.0371     0.2007  +0.2378

Lead       t2m_annual_m   t2m_max_dail   t2m_max_hour       t2m_mean   t2m_min_dail   t2m_min_hour   precip_annua   precip_max_d    precip_mean
----------------------------------------------------------------------------------------------------------------------